# Exploratory Data Analysis

## Intro
In this notebook, our goal is to verify the integrity and data quality of our data. We'll look for things such as:
1. **Sanity Check Joins**. Write a quick query to `LEFT JOIN` the fact table to all our dimensions and look for `NULL` values.
2. **ASEAN Filter**. See if dimensional tables contain a `ASEAN` block ID.
3. **Granularity**. Group the data by the desired granularity.
4. **SH vs CUCI Nomenclature**. Check differences, decide upon best way to label data.

## 1. Sanity Checking the Country Dimension (`dim_pais`)
Goal: ensure every country code in our historical fact table exists in our dimension dictoinary. If the government exported goods to a "ghost" country, we need to know how much money is missing a name.

In [2]:
import duckdb
import pandas as pd

In [3]:
# Open DuckDB connection
con = duckdb.connect()
print("DuckDB Lab open!")

DuckDB Lab open!


In [4]:
query = """SELECT * FROM 'data/silver/fact_exports.parquet' LIMIT 10"""
df_test = con.execute(query).df()
df_test

,co_ano,co_mes,co_ncm,co_unid,co_pais,sg_uf_ncm,co_via,co_urf,qt_estat,kg_liquido,vl_fob
0,2019,12,70193900,10,097,SP,07,0147600,0,0,24
1,2019,12,85444200,10,063,SP,04,0817700,99,96,9299
2,2019,12,84099111,11,493,SP,04,0817600,2,2,63
3,2019,12,87087090,11,493,SP,04,0817700,15,231,908
4,2019,12,32041990,10,586,SC,07,0147800,1200,1200,26494
5,2019,12,40169300,10,275,PR,01,0917800,2,2,103
6,2019,12,85044021,11,249,SP,04,0817700,229,51,1941
7,2019,12,21069090,10,249,SP,04,0817600,62108,62108,711535
8,2019,11,40169300,10,249,SC,04,0817600,439,439,2171
9,2019,11,61083100,11,586,SC,07,0917500,26592,3423,48414


In [5]:
# Left anti-join for countries
query_countries = """
    SELECT
        fact.co_pais,
        SUM(fact.vl_fob) AS total_value_usd,
        COUNT(*) AS row_count
    FROM 'data/silver/fact_exports.parquet' AS fact
    LEFT JOIN 'data/silver/dim_pais.parquet' AS dim
        ON fact.co_pais = dim.co_pais
    WHERE dim.no_pais IS NULL
    GROUP BY fact.co_pais
    ORDER BY total_value_usd DESC;
"""

print("Checking for missing country codes.")
df_missing_countries = con.execute(query_countries).df()

df_missing_countries

Checking for missing country codes.


,co_pais,total_value_usd,row_count


An empty dataframe on a left anti-join means our data has 100% referential integrity for country dimension. This is a very good result. Let's move on.

## 2. Sanity Checking the NCM Dimension
Goal: Ensure every product code (`dim_ncm`) in the 2019–2026 fact table exists in our current NCM dictionary.

In [6]:
# Left anti-join for NCM codes.
query_ncm = """
    SELECT
        fact.co_ncm,
        SUM(fact.vl_fob) AS total_value_usd,
        COUNT(*) AS row_count,
        MIN(fact.co_ano) AS first_year_seen,
        MAX(fact.co_ano) AS last_year_seen
    FROM 'data/silver/fact_exports.parquet' AS fact
    LEFT JOIN 'data/silver/dim_ncm.parquet' AS dim
        ON fact.co_ncm = dim.co_ncm
    WHERE dim.co_ncm IS NULL
    GROUP BY fact.co_ncm
    ORDER BY total_value_usd DESC;
"""

print("Checking for missing NCM codes.")
df_missing_ncm = con.execute(query_ncm).df()

# Check how much money is unmapped and show the top 5 missing codes.
total_missing_value = df_missing_ncm["total_value_usd"].sum()
print(f"Total unmapped USD: ${total_missing_value:.2f}")

df_missing_ncm.head()

Checking for missing NCM codes.
Total unmapped USD: $0.00


,co_ncm,total_value_usd,row_count,first_year_seen,last_year_seen


Another win for us. No unmapped exports.

## 3. Granularity Check
### 3.1 Exploring Country Blocks.
Goal: explore the `dim_pais_bloco` table to see if there is a predefined code for ASEAN block. If so, we will use it to dynamically filter our fact table.

In [7]:
query_blocks = """
    SELECT
        DISTINCT co_bloco,
        no_bloco_ing,
        no_bloco
    FROM 'data/silver/dim_pais_bloco.parquet'
    ORDER BY no_bloco;
"""

print("Showing all Economic Blocks tracked by Brazil.")

df_blocks = con.execute(query_blocks).df()

df_blocks

Showing all Economic Blocks tracked by Brazil.


,co_bloco,no_bloco_ing,no_bloco
0,105,Central America and Caribbean,América Central e Caribe
1,107,North America,América do Norte
2,48,South America,América do Sul
3,53,Association Of Southeast Asian Nations (ASEAN),Associação de Nações do Sudeste Asiático - ASEAN
4,112,Europe,Europa
5,111,Southern Common Market (MERCOSUL),Mercado Comum do Sul - Mercosul
6,61,Oceania,Oceania
7,41,Middle East,Oriente Médio
8,22,European Union (EU),União Europeia - UE
9,51,Africa (minus MIDDLE EAST),África


Result: we have a `co_bloco` for ASEAN, which is `53`. Let's now use a predicate pushdown. We are going to pre-filter `co_bloco` to decrease the size of our tables. This is an optimization technique.

In [8]:
query_asean = """
    WITH asean_block AS (
        -- Pre-filter the table
        SELECT 
            co_pais,
            no_bloco_ing
        FROM 'data/silver/dim_pais_bloco.parquet'
        WHERE co_bloco = 53
    )
    SELECT
        a.co_pais,
        p.no_pais_ing AS country_name
    FROM asean_block a
        INNER JOIN 'data/silver/dim_pais.parquet' p
        ON a.co_pais = p.co_pais
    ORDER BY p.no_pais_ing;
"""

print("Filtering for ASEAN countries...")
df_asean = con.execute(query_asean).df()

# Convert to a list of codes we can use for our gold table filter

asean_country_codes = df_asean["co_pais"].tolist()

print(f"Found {len(asean_country_codes)} ASEAN countries.")
df_asean

Filtering for ASEAN countries...
Found 10 ASEAN countries.


,co_pais,country_name
0,108,Brunei
1,141,Cambodia
2,365,Indonesia
3,420,Laos
4,455,Malaysia
5,093,Myanmar
6,267,Philippines
7,741,Singapore
8,776,Thailand
9,858,Vietnam


Great! All ASEAN countries are present and correctly mapped.

## 4. SH vs CUCI Diumensions
Goal: filter our ASEAN countries, map the exports up to the 6-digit level, and see how much we can compress the data.

In [20]:
query_sh6_test = """
    WITH asean_exports AS (
        -- Filter fact table for ASEAN countries (co_bloco = 53)
        SELECT 
            fact.co_ncm,
            fact.vl_fob
        FROM 'data/silver/fact_exports.parquet' AS fact
        INNER JOIN 'data/silver/dim_pais_bloco.parquet' AS bloco
            ON fact.co_pais = bloco.co_pais
        WHERE bloco.co_bloco = 53
    ),
    mapped_to_sh6 as (
        -- Map the NCM to the SH6 code using our dimension dict
        SELECT
            dim_ncm.co_sh6,
            SUM(e.vl_fob) AS total_exported_usd
        FROM asean_exports e
        LEFT JOIN 'data/silver/dim_ncm.parquet' AS dim_ncm
            ON e.co_ncm = dim_ncm.co_ncm
        GROUP BY dim_ncm.co_sh6
    )
    -- Bring in the SH6 names
    SELECT
        m.co_sh6,
        sh.no_sh6_ing AS product_description,
        m.total_exported_usd
    FROM mapped_to_sh6 m
    LEFT JOIN 'data/silver/dim_ncm_sh.parquet' AS sh
        ON m.co_sh6 = sh.co_sh6
    ORDER BY m.total_exported_usd DESC
    LIMIT 10
"""

print(f"Aggregating ASEAN exports to SH6 level...")
df_sh6_top10 = con.execute(query_sh6_test).df()
df_sh6_top10

Aggregating ASEAN exports to SH6 level...


,co_sh6,product_description,total_exported_usd
0,271019,"Medium oils and preparations, of petroleum or ...",2.524566e+10
1,230400,"Oilcake and other solid residues, whether or n...",1.981181e+10
2,260111,Non-agglomerated iron ores and concentrates (e...,1.520710e+10
3,270900,Petroleum oils and oils obtained from bitumino...,1.422864e+10
4,120190,"Soya beans, whether or not broken, except for ...",1.218560e+10
5,170114,Other cane sugar,8.022603e+09
6,520100,"Cotton, neither carded nor combed",7.190289e+09
7,100590,Maize (excl. seed),7.165881e+09
8,020329,Frozen meat of swine (excl. carcases and half-...,3.614617e+09
9,020714,Frozen cuts and edible offal of fowls of the s...,3.148766e+09


In [23]:
# Check the total unique rows for SH6
query_count = """
    SELECT
        COUNT(DISTINCT dim_ncm.co_sh6) AS unique_sh6_categories
    FROM 'data/silver/fact_exports.parquet' AS fact
    INNER JOIN 'data/silver/dim_pais_bloco.parquet' AS bloco
        ON fact.co_pais = bloco.co_pais
    LEFT JOIN 'data/silver/dim_ncm.parquet' AS dim_ncm
        ON fact.co_ncm = dim_ncm.co_ncm
    WHERE bloco.co_bloco = 53;
"""

row_compression = con.execute(query_count).fetchone()[0]

print(f"Total unique SH6 categories exported to ASEAN: {row_compression}")

Total unique SH6 categories exported to ASEAN: 3686


# 5. Conclusion  

Reviewing our goals:
- **Sanity Check Joins**: Completed. We ran the Left Anti-Join and proved we have $0.00 of orphaned data. Our referential integrity is 100%.
-** ASEAN Filter**: Completed. We found co_bloco = 53 and used Predicate Pushdown to cleanly isolate our 10 target countries without hardcoding them.
- **Granularity**: Completed. We tested the 6-digit level and confirmed it provides the perfect balance of executive-level readability and data compression.
- **SH vs CUCI Nomenclature**: Completed. By looking at the English SH6 output (e.g., "Frozen meat of swine"), we confirmed that the Harmonized System gives us exactly the business-friendly descriptions we need, effectively eliminating the need to bring in the CUCI dictionary.